# **Recurrent Language Modeling**
<img src="https://static01.nyt.com/images/2018/05/15/arts/01hal-voice1/merlin_135847308_098289a6-90ee-461b-88e2-20920469f96a-superJumbo.jpg" width=20% style="border-radius:20px"><br>
Let's teach computer to speak... (cause what can go wrong?)

## Table Of contents
- What is this notebook about?
- What are RNNs?
- Character-level RNN LM Implementation
- What are LSTMs and GRUs?
- Character-level LSTM and GRU LMs Implementation
- Language Modeling fun (cause why not?)
- Word Level language Modeling
- Word embeddings
- Word-level RNN LM Implementation
- Word-level LSTM and GRU LMs Implementation
- More fun!
- **Tiny Stories** and Finita La Comedia!

### What is this notebook about?
This is an introductory-level guide to language modeling with RNNs and LSTMs. and GRUs!<br>
In this one my friends we will not only discuss all the details of Recurrent Language Modeling but we will also apply it to interesting tasks, such as:
- Novel writing
- Programming (oh no... AGI takes my job!)
- LaTex
- Story Extending (Balabebe)

**Sounds Interesting? Good!**

### What are RNNs?
RNN - Recurrent Neural Network - special framework (type of neural networks), which processes input sequentially sharing all the parameters.<br>
It applies the same function to each sequence token and carries information about the past in special **hidden state**.<br>
There are various RNN visualizations, but find all of the quite confusing, because they either express nothing or give wrong ideas about practical implementation.<br>
On the image below you see a visualization of RNN wrapped out in time.<br>
<img src="https://i.imgur.com/Iveichs.jpg" style="border-radius: 20px" width=30%><br>
As you can see on each timestep RNN (green block) processes an input (character in this case) and a hidden state (it comes from left to right through the time).<br>
On each time step RNN returns an output token (at timestep t=1 it recieves input "h" and returns output "e". On the scheme visually it goes up) and updates hidden state (this one goes to the right).<br>
On intuitive level it learns to predict the next token given input token and using the previous knowledge stored in hidden state.<br>
**Important technical note!**
RNN is not a set of individual blocks, it is a one Module, which is called multiple times accumulating gradients.<br>
In other words, on practice it's more like this:<br>
<img src="https://colah.github.io/posts/2015-08-Understanding-LSTMs/img/RNN-rolled.png" width=10%><br>
But this scheme explaines nothing at all.<br>

**Important sanity note!**
RNNs are not only used for Language Modeling. They can play a role of an encoder for classification/regression tasks, they are used to work with time-series data. They are capable of more things!

But what about backpropagation? How is it possible?<br>
Well, it's called **"Backpropagation through time"** and for a good reason.<br>
At each timestep we calculate loss. We take average loss over timesteps and backpropagate from it.<br>
<img src="https://wikidocs.net/images/page/160068/7_Backpropagation-in-RNNs.jpg" style="border-radius:20px" width=30%><br>

### Character-level RNN LM Implementation

In [1]:
import torch
from torch import nn, optim
import torch.nn.functional as F
from tqdm import tqdm

import numpy as np

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [3]:
# Path is "../../Data/NLP/onegin.txt"
def load_preprocess(path, chunk_size=256, batch_size=8):
    with open(path, "r") as f:
        data = f.read()
    
    vocab = sorted(set(data)) + ["<", ">"]
    vocab_size = len(vocab)

    itos = {i:s for i, s in enumerate(vocab)}
    stoi = {s:i for i, s in itos.items()}

    chunks = [data[i:i+chunk_size] for i in range(0, len(data)-chunk_size, chunk_size)]
    chunks = ["<" + c + ">" for c in chunks]
    chunks = [[stoi[ch] for ch in chunk] for chunk in chunks]
    input = [ch[:-1] for ch in chunks]
    target = [torch.tensor(ch[1:], dtype=torch.long) for ch in chunks]
    input = [F.one_hot(torch.tensor(inp), vocab_size).float() for inp in input]

    inp_batches = [input[i:i+batch_size] for i in range(0, len(input)-batch_size, batch_size)]
    trg_batches = [target[i:i+batch_size] for i in range(0, len(target)-batch_size, batch_size)]

    inp_matrix = torch.stack([torch.stack(batch, dim=0) for batch in inp_batches], dim=0)
    trg_matrix = torch.stack([torch.stack(batch, dim=0) for batch in trg_batches], dim=0)

    return inp_matrix, trg_matrix, vocab, vocab_size, itos, stoi

In [4]:
class CharacterRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, batch_size=8):
        super().__init__()
        self.hidden_size = hidden_size
        self.batch_size = batch_size

        self.input_to_hidden = nn.Linear(input_size + hidden_size, hidden_size)
        self.activation = nn.Tanh()
        self.hidden_to_output = nn.Linear(hidden_size, output_size)
    
    def init_hidden(self, device, inference=False):
        if inference:
            return torch.zeros(1, self.hidden_size, device=device)
        else:
            return torch.zeros(self.batch_size, self.hidden_size, device=device)

    def forward(self, input, hidden):
        # Input shape: (8, 147)
        # Hidden shape: (8, hidden_size)
        # Concat along -1: (8, 147+hidden_size)
        input_with_hidden = torch.cat([input, hidden], dim=-1)
        new_hidden = self.activation(self.input_to_hidden(input_with_hidden))
        output = self.hidden_to_output(new_hidden)
        return output, new_hidden

In [5]:
from tqdm import tqdm

In [6]:
def generate(model, starter, stoi, itos, vocab_size):
    sequence = starter

    with torch.inference_mode():
        hidden = model.init_hidden(device, inference=True)
        while True:
            input = F.one_hot(torch.tensor([stoi[sequence[-1]]]), vocab_size).float().to(device)
            output, hidden = model(input, hidden)
            output_char = itos[torch.multinomial(torch.softmax(output, dim=1), 1).item()]
            if output_char == ">":
                break
            sequence += output_char
    return sequence

In [7]:
def train(model, criterion, optimizer, file_path, batch_size=8, epochs=20):
    input, target, vocab, vocab_size, itos, stoi = load_preprocess(file_path, batch_size=batch_size)
    for epoch in range(epochs):
        for input_batch, target_batch in tqdm(zip(input, target)):
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)
            
            loss = 0
            hiddens = model.init_hidden(device)
            for t in range(input_batch.shape[1]):
                character_t = input_batch[:, t, :]
                target_t = target_batch[:, t]

                predictions_t, hiddens = model(character_t, hiddens)
                loss_t = criterion(predictions_t, target_t)
                loss += loss_t
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if ((epoch+1) % 5) == 0:
            print(f"Epoch: {epoch+1} | Loss: {loss.item()}")
            print(generate(model, "<", stoi, itos, vocab_size))
    
    return model, stoi, itos, vocab_size

In [9]:
path = "../../Data/NLP/onegin.txt"
model = CharacterRNN(147, 512, 147).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
model, stoi, itos, vocab_size = train(model, criterion, optimizer, path, epochs=50)

80it [00:06, 12.07it/s]
80it [00:06, 12.30it/s]
80it [00:06, 12.38it/s]
80it [00:06, 12.30it/s]
80it [00:06, 12.25it/s]


Epoch: 5 | Loss: 609.1139526367188
<еним янь бежгоратвяхоми поеньныю слк викцасчов и кама пругоадороветет
ы Митынай
А!).8eVIIV
Дивотт, х дучел инечи. .hи лах нек о нае шыiанне, 


80it [00:06, 12.17it/s]
80it [00:06, 12.23it/s]
80it [00:06, 12.02it/s]
80it [00:06, 12.38it/s]
80it [00:06, 12.17it/s]


Epoch: 10 | Loss: 547.8370971679688
<тах-ны деянья,
К к (Не трязний перные пранита бися;
Бежиль;
Чно номины.
А6.
Бтопор о снудса
Как стмяться, ва шускеат Былжеца вераная смитьле себыю прсвей огратенще ны, Тао песнута деревсебриз лечет, нноедит отразновозды, во жчетьй:
Мна


80it [00:06, 12.14it/s]
80it [00:06, 12.20it/s]
80it [00:06, 12.16it/s]
80it [00:06, 12.18it/s]
80it [00:06, 12.22it/s]


...

Epoch: 40 | Loss: 458.8812255859375
<й-ти быск.
140
XLII

Онегин в цамей вестраянны
Прощрачных и тно нат. Он пос пяслую к


80it [00:06, 11.92it/s]
80it [00:06, 12.24it/s]
80it [00:06, 12.08it/s]
80it [00:06, 12.07it/s]
80it [00:06, 12.26it/s]


Epoch: 45 | Loss: 438.67315673828125
<рик пойнем, эти в


80it [00:06, 12.29it/s]
80it [00:06, 12.31it/s]
80it [00:06, 12.25it/s]
80it [00:06, 12.21it/s]
80it [00:06, 12.24it/s]


Epoch: 50 | Loss: 426.116943359375
<ебирет,
Вагровы погдаший сыл кругетвиих мтей неперь и взузи
В тершик за премегласьною пирит Тотьяна. Мертанога ним грези проказдца?
«Бессластной длазней но бить доскаюНи учец ее папил.
Скастаюд мне и гиде геюдостно знуют и полог
Острый продцася. Те сраза слюде двах увсчит нохосталися плозы,
За нечестаны дрядье,
Чипыт я Ланил пред ним
Встысалас фвек из песером
Непечвях меся напрадглазгиступести погдежнбым подких Ко сразбрам —
Мочьях, и кактые Евгений
Розяжбым роспокнечется
Оне IXXXII

Прыд жудна ба жезна слота
И выгом извотрев письмя вуш уж


In [64]:
print(generate(model, "О", stoi, itos, vocab_size))

Объязы но поднох, хать —
Пудя бесцые жизит, геседужнень рывадалг?
Бега думсникм роРища.. — И брежной Он сьствно, — все смечет,
Все давь гдрогий весел.
Мой девить перпол пох детрим
К о муж? Кукавью, вдобненеет уж, ненчиранцымк.
И далгою Левстик. Пустою горос, — Но, замятол;
И седь хопьят госубый поть,
Сибы пырку с празниманос
Мыняться начистречного лутшив стот и Тась и нас дебяюь —
Хоры шудя огдат, и про на,
Элеги перабесстиры не хочил..
Тонци. лпобалиные квоту,
Когда без доягча-тох
Хланий дустых тог, приведини м гдеперк ужщет онага
Чигать нак.
*

Но тразка беговереть,
Но пречен окажин багразитре) он?
Нек етдаваньем реж роф.
XXVIII

К головашит привналил: восьеший (искорю свстаянОтсконеем,
Агрымы Одегы, май удалиец
Накрожней Последе ногосЕе сталит
Тогда то что-нуб досвлупат!
Где хотк-пи былскою зупира!).
До вочих Онягин? —
«Вограсе нсказдя-тетреп,
В единих мос визеная мче:
Медпився Онегина резянальных жиетв, на. Миждо безлинаховдо нипотрила.
«С Мух госбежоненкодь сегда сох угоспеней пом

In [66]:
class CharacterDeepRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, batch_size=16):
        super().__init__()
        self.hidden_size = hidden_size
        self.batch_size = batch_size

        self.input_to_hidden1 = nn.Linear(input_size + hidden_size, hidden_size)
        self.activation1 = nn.ReLU()
        self.hidden1_to_hidden2 = nn.Linear(hidden_size * 2, hidden_size)
        self.activation2 = nn.ReLU()
        self.hidden2_to_hidden3 = nn.Linear(hidden_size * 2, hidden_size)
        self.activation3 = nn.ReLU()
        self.hidden3_to_output = nn.Linear(hidden_size, output_size)
    
    def init_hidden(self, device, inference=False):
        if inference:
            return [torch.zeros(1, self.hidden_size, device=device) for _ in range(3)]
        else:
            return [torch.zeros(self.batch_size, self.hidden_size, device=device) for _ in range(3)]

    def forward(self, input, hiddens):
        h1, h2, h3 = hiddens

        input_with_hidden1 = torch.cat([input, h1], dim=-1)
        new_hidden1 = self.activation1(self.input_to_hidden1(input_with_hidden1))

        hidden1_with_hidden2 = torch.cat([new_hidden1, h2], dim=-1)
        new_hidden2 = self.activation2(self.hidden1_to_hidden2(hidden1_with_hidden2))

        hidden2_with_hidden3 = torch.cat([new_hidden2, h3], dim=-1)
        new_hidden3 = self.activation3(self.hidden2_to_hidden3(hidden2_with_hidden3))

        output = self.hidden3_to_output(new_hidden3)
        return output, [new_hidden1, new_hidden2, new_hidden3]

In [ ]:
def train(model, criterion, optimizer, file_path, batch_size=16, epochs=20):
    input, target, vocab, vocab_size, itos, stoi = load_preprocess(file_path, batch_size=batch_size)
    for epoch in range(epochs):
        for input_batch, target_batch in tqdm(zip(input, target)):
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)
            
            loss = 0
            hiddens = model.init_hidden(device)
            for t in range(input_batch.shape[1]):
                character_t = input_batch[:, t, :]
                target_t = target_batch[:, t]

                predictions_t, hiddens = model(character_t, hiddens)
                loss_t = criterion(predictions_t, target_t)
                loss += loss_t
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if ((epoch+1) % 5) == 0:
            print(f"Epoch: {epoch+1} | Loss: {loss.item()}")
            print(generate(model, "<", stoi, itos, vocab_size))
    
    return model, stoi, itos, vocab_size

In [70]:
path = "../../Data/NLP/onegin.txt"
model1 = CharacterDeepRNN(147, 512, 147).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model1.parameters(), lr=0.001)
model1, stoi, itos, vocab_size = train(model1, criterion, optimizer, path, epochs=50)

40it [00:06,  6.50it/s]
40it [00:06,  6.55it/s]
40it [00:06,  6.56it/s]
40it [00:06,  6.51it/s]
40it [00:06,  6.46it/s]


Epoch: 5 | Loss: 680.0519409179688
<жер выйпдус .
С аоФти пдыма вву,..Ни е
1Ух неры
Яо 


40it [00:06,  6.46it/s]
40it [00:06,  6.47it/s]
40it [00:06,  6.55it/s]
40it [00:06,  6.46it/s]
40it [00:06,  6.48it/s]


Epoch: 10 | Loss: 604.2990112304688
<
Броболью.
Чреч броги уны


40it [00:06,  6.43it/s]
40it [00:06,  6.51it/s]
40it [00:06,  6.53it/s]
40it [00:06,  6.48it/s]
40it [00:06,  6.48it/s]


...

Epoch: 40 | Loss: 465.3092956542969
<ал мне прарки,
Чпостьвень


40it [00:06,  6.52it/s]
40it [00:06,  6.54it/s]
40it [00:06,  6.54it/s]
40it [00:06,  6.50it/s]
40it [00:06,  6.56it/s]


Epoch: 45 | Loss: 444.9762268066406
<им своо вЬ
Peirerarui deui, averaiiatiiui deui aneraiais deuiniterarianserileeirarersaiur,raneirerirerareiauaireriaur deuiraiviesaraieureserureriaiaa, aveureirereirerilesiraria, pareirasirerarersarerilir deriiaseaiaus derirasaa bebucriiua deiinaira augureriiuieraneur denilaianaiaieseieui,,iareraiaureriuiniuunaresraieireraiui belilarareiaisureriraraneiaueireseaasireuiuiuuisaieraiaiaiaiaurerimaniuairananeiaieiaisireriraraieureresiais aieireraraieireraiaurerieuregeai iugireriheraianaiaieiseruraranaisirerireriaasaraieirerireriieur daux lerireriheraraiauairirerianan i ileraiereieur deuinairiaiai i iuniresiheuiairereraiessieur diliairirerenairerireriiaus deniliresieur derilaiiudereraiaireriherairaraieurereraieiharsaiua deucrairasiiurerihiessiaur lieireriais, e bloraisirareriresuresiaisirarerabaureriearer iuisaenaisarirerianeraieurerieiraraaureuresuaisirerireriheui, angieur deuilaireiantireriaasirareriieii deuserireriiavaisireriraraiserireriaur derirasaass

40it [00:06,  6.48it/s]
40it [00:06,  6.44it/s]
40it [00:06,  6.48it/s]
40it [00:06,  6.51it/s]
40it [00:06,  6.29it/s]

Epoch: 50 | Loss: 452.3570251464844
<ной
Они модется пайять 


In [88]:
print(generate(model1, "О", stoi, itos, vocab_size))

Он, Лирит пельс.
LIII

На каждого лерела хлядные
А томнянный страданствивам,
В борот хожестное лучный
Изминят: Сумки билкованье
Ее одновидеть уе вечеркать
Девующ в задух, приодит»,
Подумала шомили сперок двор!
Каком! — и вперкалась... Ноша зимуются
И вопруга преприденяма
Уж укранные крепетная так ненбитретещит,
Гердесь глубонею.
XXVIX

Ктемь? КтОробующие.
Брагоблюпыл он клиняться
Продействе радая
Сихать. Руссовой,
Грепобу проворять,
Улибкоррямы давно,
На старому предлюбленным изменит.
44
Потыблая предресное дверу...»
XXXVIIII

Чтно согрязя, Буять селевам.
XXXV

Как пред думи черпить:
Как ужачною перен
Расставшества старою темные почилась
И, умелела опоставенный сон.
162
XIX
 ин устареной
Занящистью готовины
Ее теправдушков золумебляя
Бавчушным паружках.
Беспременая знакки чевстретельных сне


In [94]:
class CharacterLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, batch_size=16):
        super().__init__()
        self.hidden_size = hidden_size
        self.batch_size = batch_size

        self.f_t = nn.Linear(input_size + hidden_size, hidden_size)
        self.f_t_gate = nn.Sigmoid()

        self.i_t = nn.Linear(input_size + hidden_size, hidden_size)
        self.i_t_gate = nn.Sigmoid()
        self._c_t = nn.Linear(input_size + hidden_size, hidden_size)
        self._c_t_act = nn.Tanh()

        self.o_t = nn.Linear(input_size + hidden_size, hidden_size)
        self.o_t_gate = nn.Sigmoid()
        self.c_t_act = nn.Tanh()

        self.output = nn.Linear(hidden_size, output_size)
    
    # Actually, should've better called it init_states, but I'm
    # Constrained with training function name assumptions.
    def init_hidden(self, device, inference=False):
        if inference:
            return [torch.zeros(1, self.hidden_size, device=device) for _ in range(2)]
        else:
            return [torch.zeros(self.batch_size, self.hidden_size, device=device) for _ in range(2)]

    def forward(self, input, states):
        hid, cell = states

        input_with_hidden = torch.cat([input, hid], dim=-1)

        f_t = self.f_t_gate(self.f_t(input_with_hidden))

        i_t = self.i_t_gate(self.i_t(input_with_hidden))
        _c_t = self._c_t_act(self._c_t(input_with_hidden))

        updated_cell = f_t * cell + i_t * _c_t

        o_t = self.o_t_gate(self.o_t(input_with_hidden))
        new_hidden = o_t * self.c_t_act(updated_cell)

        output = self.output(new_hidden)

        return output, [new_hidden, updated_cell]


In [190]:
def train(model, criterion, optimizer, file_path, batch_size=8, epochs=20):
    input, target, vocab, vocab_size, itos, stoi = load_preprocess(file_path, batch_size=batch_size)
    for epoch in range(epochs):
        for input_batch, target_batch in tqdm(zip(input, target)):
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)
            
            loss = 0
            hiddens = model.init_hidden(device)
            for t in range(input_batch.shape[1]):
                character_t = input_batch[:, t, :]
                target_t = target_batch[:, t]

                predictions_t, hiddens = model(character_t, hiddens)
                loss_t = criterion(predictions_t, target_t)
                loss += loss_t
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if ((epoch+1) % 10) == 0:
            print(f"Epoch: {epoch+1} | Loss: {loss.item()}")
            print(generate(model, "<", stoi, itos, vocab_size))
    
    return model, stoi, itos, vocab_size

In [191]:
path = "../../Data/NLP/onegin.txt"
model2 = CharacterLSTM(147, 512, 147, batch_size=16).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model2.parameters(), lr=0.001)
model2, stoi, itos, vocab_size = train(model2, criterion, optimizer, path, epochs=100, batch_size=16)

40it [00:07,  5.18it/s]
40it [00:07,  5.08it/s]
40it [00:07,  5.05it/s]
40it [00:07,  5.07it/s]
40it [00:07,  5.23it/s]
40it [00:07,  5.17it/s]
40it [00:07,  5.13it/s]
40it [00:07,  5.10it/s]
40it [00:07,  5.16it/s]
40it [00:07,  5.06it/s]


Epoch: 10 | Loss: 656.4967041015625
<Bбой пуч, 1л.
Не ень  птизбувс. . К поу мно . н и. дулесtва .
С сто, — зе мена . . г в но мо б не ный,мПрану1,,Шат на стивnк
е  нем,
9.

 бинино ррутта генестСвнодн.
И д мс4.
И счя ми.

Ны реск.г)
Ни . ( зани днай омри дкохластенит зСтальтомегне в мнеся Зазпя.Прачныхост ригир пронят,
Плорожкомо нтя ш ссли
на итзоб!..
IIIЕ
Вмлvень дér. 4 


40it [00:07,  5.12it/s]
40it [00:07,  5.12it/s]
40it [00:07,  5.14it/s]
40it [00:07,  5.15it/s]
40it [00:07,  5.22it/s]
40it [00:07,  5.14it/s]
40it [00:07,  5.17it/s]
40it [00:07,  5.15it/s]
40it [00:07,  5.14it/s]
40it [00:07,  5.11it/s]


Epoch: 20 | Loss: 598.540771484375
<ко Тидей змугсе прежнайвить Одожный Мвез естатле
Татркуй в зим ут.
XXXL

Потней плюблю вого Мы да.
Гто
Вгяемиснег, дю: зхика. 
За дава ко


40it [00:07,  5.09it/s]
40it [00:07,  5.17it/s]
40it [00:07,  5.14it/s]
40it [00:07,  5.16it/s]
40it [00:07,  5.15it/s]
40it [00:07,  5.14it/s]
40it [00:07,  5.15it/s]
40it [00:07,  5.13it/s]
40it [00:07,  5.18it/s]
40it [00:07,  5.14it/s]


...

Epoch: 80 | Loss: 442.4416198730469
< мне кочемами
Благон, нах делся, замаланы,
Его взепялвимися промал,
Завети ж ятчиского забы.
XXI

Стою чашуютсю измелана
Сердиб А соном полеблюбонно
С остахом утражелеск устревных,
Вы праззашают прокрымал
Ярил баз грубов кробиков
С душают вежно прежниковми
Вге


40it [00:07,  5.20it/s]
40it [00:07,  5.18it/s]
40it [00:07,  5.20it/s]
40it [00:07,  5.17it/s]
40it [00:07,  5.20it/s]
40it [00:07,  5.19it/s]
40it [00:07,  5.19it/s]
40it [00:07,  5.19it/s]
40it [00:07,  5.16it/s]
40it [00:07,  5.18it/s]


Epoch: 90 | Loss: 410.41290283203125
<ол, вирела одна
Я накосе дево не ума:
Одным к Вервие кнась в готию,
И придоскость.... высь двороми
И стала призднака, пере.
А ту, к чась пыстро ручкоеет;
И, перневнями обкажает;
Сидают неще ничтинить
Сем насходли блус


40it [00:07,  5.20it/s]
40it [00:07,  5.14it/s]
40it [00:07,  5.20it/s]
40it [00:07,  5.22it/s]
40it [00:07,  5.24it/s]
40it [00:07,  5.13it/s]
40it [00:07,  5.21it/s]
40it [00:07,  5.25it/s]
40it [00:07,  5.21it/s]
40it [00:07,  5.20it/s]


Epoch: 100 | Loss: 366.5318603515625
<м съезжастви; сликаховы,
Когда в окамленный крас
Эля жизнь я. коклялость,
Такия барвы свит небра.
СLDc пмучте выхольна рока,
Как первый ё ного нобосал,
И верно рида скол так на смеда,
За любом екати сесейся.
XXXVIII

Быта маж там недрастворогом.
В весецомен в по


In [238]:
print(generate(model2, "О", stoi, itos, vocab_size))

О1).
XXVII


Гmo insнe à bonné н


In [247]:
print(generate(model2, "<", stoi, itos, vocab_size)[1:])

уку
Проливы предана; чевеца
14
Но правдал после предустая!
Та смел нас когояниться льменству,
Приведной душные досы,
Разгровлив, мы знаем друзья?
Увыло но пуня в томоре;
Мо верпекахом пыкакам.
58
XXI

Вог нек правожном свои неспрова;
Меняльном полкною провидь
Призялся, здестьющей нед,
Пукрый гтрас в полеку ризногу,
Гот вардит в тином суких.
XXII

Тае дригочиватся, всём уселе
Смергий половина снег;
ЕXIII

И сепрался не дом менцых сут,
Пода нег сердцем опеслевы;
И скаро пилых тай печловоные
На грубстой содка: в похмуренной
У нах нелгаховакЕвгений?
Мне смарткт, молоды, хлачает,
Певших постовь мадорнивог;
144
Уж еед снугание умой
В сем не зачем в данцуго.
XXVII

Оламине, благослише, вмновь.
Так теско льют, и мороченный,
Как минит икому свою:
И дон ведни возвы златавит,
Таперь он все шевет, и себит:
Он сердце не навих бог,
Девца их наши лет,
Мыельном вырон и страстет
Скучно вростны: муж
